# Consumer: Model Review & Approval

This notebook simulates a downstream consumer that:
1. Authenticates with MinIO
2. Polls for models awaiting approval
3. Downloads and evaluates the model against a local test dataset
4. Uploads an approval file to MinIO, triggering promotion to production

**Prerequisites:** Configure credentials using the `.env` uploader below, or fill in the manual config cell.

## 1. Configuration

**Option A:** Upload a `.env` file — `MINIO_USER` and `MINIO_PASS` will be populated automatically.  
**Option B:** Fill in the values manually in the cell below.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# Defaults — overwritten when a .env file is uploaded
MINIO_USER = "your_minio_username"
MINIO_PASS = "your_minio_password"


def _parse_env(content: str) -> dict:
    result = {}
    for line in content.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if "=" in line:
            key, _, value = line.partition("=")
            result[key.strip()] = value.strip().strip('"').strip("'")
    return result


def _on_env_upload(change):
    global MINIO_USER, MINIO_PASS
    value = change["new"]
    # ipywidgets >=8 returns a tuple of dicts; <8 returns a dict keyed by filename
    entries = value if isinstance(value, (list, tuple)) else value.values()
    for entry in entries:
        raw = entry["content"]
        text = bytes(raw).decode("utf-8") if not isinstance(raw, str) else raw
        env = _parse_env(text)
        if "MINIO_USER" in env:
            MINIO_USER = env["MINIO_USER"]
        if "MINIO_PASS" in env:
            MINIO_PASS = env["MINIO_PASS"]
        name = entry.get("name", entry.get("metadata", {}).get("name", ".env"))
        env_status.value = (
            f"<b style='color:green'>Loaded {name}</b> — "
            f"MINIO_USER=<code>{MINIO_USER}</code>, "
            f"MINIO_PASS=<code>{'*' * len(MINIO_PASS)}</code>"
        )


uploader = widgets.FileUpload(accept=".env", multiple=False, description="Upload .env")
env_status = widgets.HTML(value="<i style='color:grey'>No .env file uploaded yet.</i>")
uploader.observe(_on_env_upload, names="value")
display(widgets.VBox([uploader, env_status]))

In [ ]:
# --- Manual config (skip if you uploaded a .env above) ---
# MINIO_USER = "your_minio_username"
# MINIO_PASS = "your_minio_password"

MINIO_BUCKET = "smart-healthcare-diabetes-models"

# --- Local test data (consumer-side) ---
# These files are owned by the consumer and are NOT fetched from MinIO.
SCALER_PATH   = "./scaler.pkl"     # joblib-serialised StandardScaler
TEST_CSV_PATH = "./test_data.csv"  # CSV with the same columns used during training

# Column names in the test CSV
TARGET_COLUMN   = "Hyperglycemia"
COLUMNS_TO_DROP = ["ShortId", "Hypoglycemia"]  # adjust or leave empty

# Approval threshold shown as a recommendation in the review widget
APPROVAL_THRESHOLD = 0.70

## 2. MinIO helpers (self-contained, no local imports needed)

In [ ]:
import json
import os
import tempfile
import requests
from datetime import datetime, timezone


def minio_auth(user, password):
    resp = requests.post(
        "https://humaine-minio-api.euprojects.net/auth/auth",
        data={"username": user, "password": password},
        headers={"Content-Type": "application/x-www-form-urlencoded"},
    )
    if resp.status_code == 200:
        return resp.json()["access_token"]
    raise RuntimeError(f"MinIO auth failed: {resp.status_code} {resp.text}")


def minio_read_json(token, bucket, object_name):
    """Return parsed JSON from MinIO object, or None if not found."""
    resp = requests.get(
        f"https://humaine-minio-api.euprojects.net/main_ops/download/{bucket}/{object_name}",
        headers={"Authorization": f"Bearer {token}", "accept": "application/json"},
    )
    return resp.json() if resp.status_code == 200 else None


def minio_download(token, bucket, object_name, file_path):
    resp = requests.get(
        f"https://humaine-minio-api.euprojects.net/main_ops/download/{bucket}/{object_name}",
        headers={"Authorization": f"Bearer {token}", "accept": "application/json"},
    )
    if resp.status_code == 200:
        with open(file_path, "wb") as f:
            f.write(resp.content)
    else:
        raise RuntimeError(f"Download failed: {resp.status_code} {resp.text}")


def minio_write_json(token, bucket, object_name, data):
    tmp = tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False)
    json.dump(data, tmp)
    tmp.close()
    with open(tmp.name, "rb") as f:
        resp = requests.post(
            "https://humaine-minio-api.euprojects.net/main_ops/upload",
            data={"bucket_name": bucket, "object_name": object_name},
            files={"file": (object_name, f, "application/json")},
            headers={"Authorization": f"Bearer {token}"},
        )
    os.unlink(tmp.name)
    if resp.status_code != 200:
        raise RuntimeError(f"Upload failed: {resp.status_code} {resp.text}")
    print("Upload successful:", resp.text)

## 3. Authenticate with MinIO

In [ ]:
token = minio_auth(MINIO_USER, MINIO_PASS)
print("Authenticated successfully.")

## 4. Poll for pending models

Reads `published-models/index.json` and picks the most recent version with `status: pending_approval`.

In [ ]:
index = minio_read_json(token, MINIO_BUCKET, "published-models/index.json")

if not index:
    print("No published models found in MinIO.")
    version_to_review = None
else:
    pending = [e for e in index if e["status"] == "pending_approval"]
    if not pending:
        print("No models awaiting approval.")
        print("Current index:")
        for e in index:
            print(f"  version={e['version']}  status={e['status']}")
        version_to_review = None
    else:
        # The index is newest-first, so pending[0] is the most recent
        version_to_review = pending[0]["version"]
        print(f"Found pending model: version={version_to_review}")
        print(f"Published at: {pending[0]['published_at']}")

## 5. Download the model from MinIO

In [ ]:
assert version_to_review is not None, "No pending model to review — re-run cell 4 after a model is published."

model_object = f"published-models/{version_to_review}/ltn.h5"
local_model_path = f"./downloaded_ltn_{version_to_review}.h5"

minio_download(token, MINIO_BUCKET, model_object, local_model_path)
print(f"Model downloaded to: {local_model_path}")

## 6. Evaluate the model against local test data

In [ ]:
import numpy as np
import pandas as pd
import joblib
from tensorflow.keras.models import load_model
from sklearn.metrics import confusion_matrix, classification_report, balanced_accuracy_score

# Load test data
df = pd.read_csv(TEST_CSV_PATH)
if COLUMNS_TO_DROP:
    df = df.drop(columns=[c for c in COLUMNS_TO_DROP if c in df.columns])

X_test = df.drop(columns=[TARGET_COLUMN])
y_test = df[TARGET_COLUMN]

# Scale
scaler = joblib.load(SCALER_PATH)
scaler.fit(X_test)
X_test_scaled = scaler.transform(X_test)

# Load model and predict
model = load_model(local_model_path)
y_pred = (model.predict(X_test_scaled) > 0.5).astype(int)

# Metrics
bal_acc = balanced_accuracy_score(y_test, y_pred)
report  = classification_report(y_test, y_pred)
cm      = confusion_matrix(y_test, y_pred)

print(f"Balanced Accuracy: {bal_acc:.4f}")
print()
print(report)
print("Confusion matrix:")
print(cm)

## 7. Review & Approve

The panel below summarises the evaluation results and shows a recommendation based on `APPROVAL_THRESHOLD`.  
Press **Approve** or **Reject** to record your decision — the approval file is uploaded to MinIO immediately on click.

In [ ]:
# ── build summary text ────────────────────────────────────────────────────────
meets_threshold = bal_acc >= APPROVAL_THRESHOLD
rec_color  = "green" if meets_threshold else "darkorange"
rec_label  = "Meets threshold" if meets_threshold else "Below threshold"

summary_html = widgets.HTML(value=f"""
<div style='font-family:monospace; border:1px solid #ccc; border-radius:6px;
            padding:12px 16px; margin-bottom:8px; background:#fafafa'>
  <b>Model version:</b> {version_to_review}<br>
  <b>Balanced accuracy:</b> {bal_acc:.4f} &nbsp;
  <span style='color:{rec_color}'><b>({rec_label} — threshold {APPROVAL_THRESHOLD})</b></span>
</div>
""")

# ── buttons ───────────────────────────────────────────────────────────────────
btn_approve = widgets.Button(
    description="Approve",
    button_style="success",
    icon="check",
    layout=widgets.Layout(width="140px"),
)
btn_reject = widgets.Button(
    description="Reject",
    button_style="danger",
    icon="times",
    layout=widgets.Layout(width="140px"),
)
decision_status = widgets.HTML(value="<i style='color:grey'>Awaiting decision…</i>")


def _do_approve(_):
    btn_approve.disabled = True
    btn_reject.disabled  = True
    decision_status.value = "<i style='color:grey'>Uploading approval…</i>"
    try:
        payload = {
            "version": version_to_review,
            "approved_at": datetime.now(timezone.utc).isoformat(),
            "balanced_accuracy": round(float(bal_acc), 4),
            "threshold_used": APPROVAL_THRESHOLD,
            "note": "approved via notebook button",
        }
        minio_write_json(token, MINIO_BUCKET, f"approvals/{version_to_review}/approval.json", payload)
        decision_status.value = (
            f"<b style='color:green'>Approved.</b> "
            f"Approval file uploaded to <code>approvals/{version_to_review}/approval.json</code>. "
            f"The Streamlit app will promote this model to production on the next Models tab visit."
        )
    except Exception as e:
        btn_approve.disabled = False
        btn_reject.disabled  = False
        decision_status.value = f"<b style='color:red'>Upload failed:</b> {e}"


def _do_reject(_):
    btn_approve.disabled = True
    btn_reject.disabled  = True
    decision_status.value = (
        f"<b style='color:darkorange'>Rejected.</b> "
        f"No approval file uploaded for version <code>{version_to_review}</code>."
    )


btn_approve.on_click(_do_approve)
btn_reject.on_click(_do_reject)

display(widgets.VBox([
    summary_html,
    widgets.HBox([btn_approve, btn_reject]),
    decision_status,
]))

## 8. Cleanup (optional)

Remove the locally downloaded model file once review is complete.

In [ ]:
if os.path.exists(local_model_path):
    os.remove(local_model_path)
    print(f"Removed local file: {local_model_path}")